In [1]:
pip install torch==1.13.1+cu116 torchvision==0.14.1+cu116 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu116

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu116
  Using cached https://download-r2.pytorch.org/whl/cu116/torch-1.13.1%2Bcu116-cp311-cp311-linux_x86_64.whl (1977.9 MB)
ERROR: Could not find a version that satisfies the requirement torchvision==0.14.1+cu116 (from versions: 0.1.6, 0.1.7, 0.1.8, 0.1.9, 0.2.0, 0.2.1, 0.2.2, 0.2.2.post2, 0.2.2.post3, 0.15.0, 0.15.1, 0.15.2, 0.16.0, 0.16.1, 0.16.2, 0.17.0, 0.17.1, 0.17.2, 0.18.0, 0.18.1, 0.19.0, 0.19.1, 0.20.0, 0.20.1, 0.21.0, 0.22.0, 0.22.1, 0.23.0, 0.24.0, 0.24.1, 0.25.0, 0.26.0)
ERROR: No matching distribution found for torchvision==0.14.1+cu116
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gradio as gr
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
import librosa
import pandas as pd

# ── Change these ───────────────────────────────────────────────────

BASE_DIR = os.path.abspath(os.getcwd())

# If running inside Finalproject/, go up one level
if os.path.basename(BASE_DIR) == "Finalproject":
    ROOT_DIR = os.path.dirname(BASE_DIR)
else:
    ROOT_DIR = BASE_DIR

CKPT_PATH = os.path.join(ROOT_DIR, "Finalproject", "best_phase2resample.pth")
TAXONOMY  = os.path.join(ROOT_DIR, "Finalproject", "taxonomy.csv")
CNN14_DIR = os.path.join(ROOT_DIR, "audioset_tagging_cnn", "pytorch")
# ────────────────────────────────────────────────────────────────────

sys.path.append(CNN14_DIR)
from models import Cnn14


class Cnn14Pantanal(nn.Module):
    def __init__(self, base_model, num_classes):
        super().__init__()
        self.spectrogram_extractor = base_model.spectrogram_extractor
        self.logmel_extractor      = base_model.logmel_extractor
        self.spec_augmenter        = base_model.spec_augmenter
        self.bn0         = base_model.bn0
        self.conv_block1 = base_model.conv_block1
        self.conv_block2 = base_model.conv_block2
        self.conv_block3 = base_model.conv_block3
        self.conv_block4 = base_model.conv_block4
        self.conv_block5 = base_model.conv_block5
        self.conv_block6 = base_model.conv_block6
        self.fc1 = nn.Sequential(base_model.fc1)
        self.gelu = nn.GELU()
        self.fc_audioset = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.spectrogram_extractor(x)
        x = self.logmel_extractor(x)
        x = x.transpose(1, 3)
        x = self.bn0(x)
        x = x.transpose(1, 3)
        x = self.conv_block1(x, pool_size=(2,2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=False)
        x = self.conv_block2(x, pool_size=(2,2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=False)
        x = self.conv_block3(x, pool_size=(2,2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=False)
        x = self.conv_block4(x, pool_size=(2,2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=False)
        x = self.conv_block5(x, pool_size=(2,2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=False)
        x = self.conv_block6(x, pool_size=(1,1), pool_type='avg')
        x = F.dropout(x, p=0.2, training=False)
        x = torch.mean(x, dim=3)
        x1, _ = torch.max(x, dim=2)
        x2    = torch.mean(x, dim=2)
        x = x1 + x2
        x = F.dropout(x, p=0.5, training=False)
        x = self.gelu(self.fc1(x))
        x = F.dropout(x, p=0.5, training=False)
        return {"clipwise_output": self.fc_audioset(x)}

TARGET_SR = 32000
CLIP_SEC  = 5
CLIP_LEN  = TARGET_SR * CLIP_SEC

taxonomy    = pd.read_csv(TAXONOMY)
labels      = taxonomy["common_name"].values
NUM_CLASSES = len(labels)

# ... (paste your Cnn14Pantanal class here) ...

# ── Load model once at startup ──────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base = Cnn14(sample_rate=32000, window_size=1024, hop_size=320,
             mel_bins=64, fmin=50, fmax=14000, classes_num=527)
model = Cnn14Pantanal(base, NUM_CLASSES).to(device)
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()

# ── Inference function ──────────────────────────────────────────────
def predict_bird(audio_path):
    audio, sr = sf.read(audio_path)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)
    audio = audio.astype(np.float32)

    clips = []
    for start in range(0, len(audio), CLIP_LEN):
        clip = audio[start : start + CLIP_LEN]
        if len(clip) < CLIP_LEN:
            clip = np.pad(clip, (0, CLIP_LEN - len(clip)))
        mx = np.abs(clip).max()
        if mx > 0:
            clip = clip / mx
        clips.append(clip)

    clips = np.stack(clips)
    with torch.no_grad():
        batch = torch.tensor(clips).to(device)
        out   = model(batch)
        probs = torch.sigmoid(out["clipwise_output"]).cpu().numpy()

    avg_probs   = probs.mean(axis=0)
    top_k_idx   = avg_probs.argsort()[::-1][:TOP_K]

    # Return a dict of {label: confidence} for Gradio's label component
    return {labels[idx]: float(avg_probs[idx]) for idx in top_k_idx}

# ── Launch the app ──────────────────────────────────────────────────
demo = gr.Interface(
    fn=predict_bird,
    inputs=gr.Audio(type="filepath", label="Upload a bird recording"),
    outputs=gr.Label(num_top_classes=TOP_K, label="Predicted Species"),
    title="🐦 BirdCLEF Species Identifier",
    description="Upload a WAV or OGG recording and the model will identify the bird species.",
)

demo.launch(server_name="0.0.0.0", share=True)

/home/users/ss1482/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-12GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/home/users/ss1482/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/home/users/ss1482/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU1 Tesla P100-PCIE-12GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/home/users/ss1482/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-12GB with CUDA capability sm_60 is not compatible w

In [4]:
import os
print(os.path.exists(CNN14_DIR))
print(os.listdir(CNN14_DIR))

False


FileNotFoundError: [Errno 2] No such file or directory: 'sangcs372final/audioset_tagging_cnn/pytorch'